In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [7]:
print(df.columns)

Index(['transaction_id', 'amount', 'transaction_hour', 'merchant_category',
       'foreign_transaction', 'location_mismatch', 'device_trust_score',
       'velocity_last_24h', 'cardholder_age', 'is_fraud'],
      dtype='object')


In [9]:
df = pd.read_csv("/content/credit_card_fraud_10k.csv")

In [10]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

In [13]:
X = pd.get_dummies(X, columns=["merchant_category"], drop_first=True)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train, y_train
)

In [15]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train_smote, y_train_smote)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [16]:
y_prob = xgb_model.predict_proba(X_test)[:, 1]

In [17]:
threshold = 0.3
y_pred = (y_prob >= threshold).astype(int)

print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


XGBoost ROC-AUC: 0.9843147208121827
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1970
           1       0.35      0.77      0.48        30

    accuracy                           0.98      2000
   macro avg       0.68      0.87      0.74      2000
weighted avg       0.99      0.98      0.98      2000



In [18]:
feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Feature Importance:")
print(feature_importance)

Feature Importance:
device_trust_score               0.206984
transaction_hour                 0.182346
velocity_last_24h                0.112798
location_mismatch                0.095390
foreign_transaction              0.093055
merchant_category_Food           0.072868
merchant_category_Travel         0.060370
merchant_category_Grocery        0.060236
merchant_category_Electronics    0.046262
transaction_id                   0.025484
amount                           0.022995
cardholder_age                   0.021214
dtype: float32


In [19]:

svm_model = SVC(
    probability=True,
    random_state=42
)

svm_model.fit(X_train_smote, y_train_smote)

svm_prob = svm_model.predict_proba(X_test)[:, 1]
svm_pred = (svm_prob >= 0.5).astype(int)

print("Baseline SVM ROC-AUC:", roc_auc_score(y_test, svm_prob))
print(classification_report(y_test, svm_pred))

Baseline SVM ROC-AUC: 0.6322673434856176
              precision    recall  f1-score   support

           0       0.99      0.52      0.68      1970
           1       0.02      0.67      0.04        30

    accuracy                           0.52      2000
   macro avg       0.51      0.59      0.36      2000
weighted avg       0.98      0.52      0.67      2000

